In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
from tqdm import tqdm
from matplotlib import pyplot as plt # show graph

from sklearn.model_selection import GroupShuffleSplit
from hmmlearn import hmm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
with open("GSD_train.txt", encoding='utf-8') as f:
  data = f.read()
sent = data.split('\n\n')

In [7]:
tokens = []
for sentence in sent:
    s = []
    for word in sentence.split('\n'):
        if word:
            token = (word.split()[1], word.split()[3])
            s.append(token)
    tokens.append(s)

In [25]:
tagged_words = [(i, *tup) for i, sent in enumerate(tokens) for tup in sent]
tagged_words

[(0, 'Начальный', 'ADJ'),
 (0, 'ролик', 'NOUN'),
 (0, ',', 'PUNCT'),
 (0, 'или', 'CCONJ'),
 (0, 'опенинг', 'NOUN'),
 (0, '(', 'PUNCT'),
 (0, 'от', 'ADP'),
 (0, ',', 'PUNCT'),
 (0, 'сокр.', 'ADV'),
 (0, ':', 'PUNCT'),
 (0, 'OP', 'X'),
 (0, ')', 'PUNCT'),
 (0, ',', 'PUNCT'),
 (0, 'как', 'ADP'),
 (0, 'правило', 'NOUN'),
 (0, ',', 'PUNCT'),
 (0, 'представляет', 'VERB'),
 (0, 'собой', 'PRON'),
 (0, 'анимацию', 'NOUN'),
 (0, ',', 'PUNCT'),
 (0, 'изображающую', 'VERB'),
 (0, 'главных', 'ADJ'),
 (0, 'героев', 'NOUN'),
 (0, 'аниме', 'NOUN'),
 (0, 'и', 'CCONJ'),
 (0, 'отражающую', 'VERB'),
 (0, 'его', 'DET'),
 (0, 'стиль', 'NOUN'),
 (0, '.', 'PUNCT'),
 (1, 'Стропило', 'NOUN'),
 (1, ',', 'PUNCT'),
 (1, 'означающее', 'VERB'),
 (1, 'победителя', 'NOUN'),
 (1, ',', 'PUNCT'),
 (1, 'окрашено', 'VERB'),
 (1, 'в', 'ADP'),
 (1, 'красный', 'ADJ'),
 (1, 'цвет', 'NOUN'),
 (1, '--', 'PUNCT'),
 (1, 'цвет', 'NOUN'),
 (1, 'гербового', 'ADJ'),
 (1, 'щита', 'NOUN'),
 (1, 'Москвы', 'PROPN'),
 (1, ',', 'PUNCT'),
 (

In [3]:
data = pd.DataFrame(tagged_words, columns=['sentence', 'Word', 'POS'])
data

NameError: name 'tagged_words' is not defined

In [4]:
data = pd.read_csv("NER dataset.csv", encoding='latin1')
data = data.fillna(method="ffill")
data = data.rename(columns={'Sentence #': 'sentence'})
data.head(5)

C:\Users\Андрей\AppData\Local\Temp\ipykernel_10532\2134620346.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method="ffill")


,sentence,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O


Get the numbers of tags & words inside the whole data. We'll need this in the future.

In [5]:
tags = list(set(data.POS.values)) #Read POS values
words = list(set(data.Word.values))
len(tags), len(words)

(42, 35177)

We cannot split data normally with `train_test_split` because doing that makes some parts of a sentence in the training set while some others in the testing set. Instead, we use `GroupShuffleSplit`.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
y = data.POS
X = data.drop('POS', axis=1)

gs = GroupShuffleSplit(n_splits=2, test_size=.33, random_state=42)
train_ix, test_ix = next(gs.split(X, y, groups=data['sentence']))

data_train = data.loc[train_ix]
data_test = data.loc[test_ix]

data_train

,sentence,Word,POS,Tag
24,Sentence: 2,Families,NNS,O
25,Sentence: 2,of,IN,O
26,Sentence: 2,soldiers,NNS,O
27,Sentence: 2,killed,VBN,O
28,Sentence: 2,in,IN,O
...,...,...,...,...
1048570,Sentence: 47959,they,PRP,O
1048571,Sentence: 47959,responded,VBD,O
1048572,Sentence: 47959,to,TO,O
1048573,Sentence: 47959,the,DT,O


After checking the data after splitted, it seems to be fine.
Check the numbers of tags & words in the training set.

In [8]:
tags = list(set(data_train.POS.values)) #Read POS values
words = list(set(data_train.Word.values))
len(tags), len(words)

(42, 29586)

The number of tags is enough but the number of words is not enough (~29k vs ~35k).
Because of that we need to randomly add some UNKNOWN words into the training dataset then we recalculate the word list and create map from them to number.

In [9]:
dfupdate = data_train.sample(frac=.15, replace=False, random_state=42)
dfupdate.Word = 'UNKNOWN'
data_train.update(dfupdate)
words = list(set(data_train.Word.values))
# Convert words and tags into numbers
word2id = {w: i for i, w in enumerate(words)}
tag2id = {t: i for i, t in enumerate(tags)}
id2tag = {i: t for i, t in enumerate(tags)}
len(tags), len(words)

(42, 27553)

Hidden Markov Models can be trained by using the Baum-Welch algorithm.
However input of the training is just dataset (Words).
We cannot map back the states to the POS tag.

That's why we have to calculate the model parameters for `hmmlearn.hmm.MultinomialHMM` manually by calculating
- `startprob_`
- `transmat_`
- `emissionprob_`

In [10]:
count_tags = dict(data_train.POS.value_counts())
count_tags_to_words = data_train.groupby(['POS']).apply(lambda grp: grp.groupby('Word')['POS'].count().to_dict()).to_dict()
count_init_tags = dict(data_train.groupby('sentence').first().POS.value_counts())

# TODO use panda solution
count_tags_to_next_tags = np.zeros((len(tags), len(tags)), dtype=int)
sentences = list(data_train.sentence)
pos = list(data_train.POS)
for i in range(len(sentences)) :
    if (i > 0) and (sentences[i] == sentences[i - 1]):
        prevtagid = tag2id[pos[i - 1]]
        nexttagid = tag2id[pos[i]]
        count_tags_to_next_tags[prevtagid][nexttagid] += 1

C:\Users\Андрей\AppData\Local\Temp\ipykernel_10532\278955984.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  count_tags_to_words = data_train.groupby(['POS']).apply(lambda grp: grp.groupby('Word')['POS'].count().to_dict()).to_dict()


In [11]:
mystartprob = np.zeros((len(tags),))
mytransmat = np.zeros((len(tags), len(tags)))
myemissionprob = np.zeros((len(tags), len(words)))
num_sentences = sum(count_init_tags.values())
sum_tags_to_next_tags = np.sum(count_tags_to_next_tags, axis=1)
for tag, tagid in tag2id.items():
    floatCountTag = float(count_tags.get(tag, 0))
    mystartprob[tagid] = count_init_tags.get(tag, 0) / num_sentences
    for word, wordid in word2id.items():
        myemissionprob[tagid][wordid]= count_tags_to_words.get(tag, {}).get(word, 0) / floatCountTag
    for tag2, tagid2 in tag2id.items():
        mytransmat[tagid][tagid2]= count_tags_to_next_tags[tagid][tagid2] / sum_tags_to_next_tags[tagid]

Initialize a HMM

In [12]:
model = hmm.CategoricalHMM(n_components=len(tags), algorithm='viterbi', random_state=42)
model.startprob_ = mystartprob
model.transmat_ = mytransmat
model.emissionprob_ = myemissionprob

As some words may never appear in the training set, we need to transform them into `UNKNOWN` first.
Then we split `data_test` into `samples` & `lengths` and send them to HMM.

In [25]:
data_test.loc[~data_test['Word'].isin(words), 'Word'] = 'UNKNOWN'
word_test = list(data_test.Word)
samples = []
for i, val in enumerate(word_test):
    samples.append([word2id[val]])
    
# TODO use panda solution
lengths = []
count = 0
sentences = list(data_test.sentence)
for i in range(len(sentences)) :
    if (i > 0) and (sentences[i] == sentences[i - 1]):
        count += 1
    elif i > 0:
        lengths.append(count)
        count = 1
    else:
        count = 1

In [26]:
help(model.predict)

Help on method predict in module hmmlearn.base:

predict(X, lengths=None) method of hmmlearn.hmm.CategoricalHMM instance
    Find most likely state sequence corresponding to ``X``.
    
    Parameters
    ----------
    X : array-like, shape (n_samples, 1)
        Feature matrix of individual samples.
    lengths : array-like of integers, shape (n_sequences, ), optional
        Lengths of the individual sequences in ``X``. The sum of
        these should be ``n_samples``.
    
    Returns
    -------
    state_sequence : array, shape (n_samples, )
        Labels for each sample from ``X``.
    
    Notes
    -----
    Unlike other HMM classes, `CategoricalHMM` ``X`` arrays have shape
    ``(n_samples, 1)`` (instead of ``(n_samples, n_features)``).  Consider using
    `sklearn.preprocessing.LabelEncoder` to transform your input to the right
    format.



In [27]:
# This code is very slow
pos_predict = model.predict(samples, lengths)
pos_predict

ValueError: lengths array [24, 21, 28, 25, 26, 22, 34, 23, 19, 25, 22, 35, 30, 24, 26, 39, 31, 7, 34, 8, 26, 32, 14, 30, 24, 11, 17, 29, 27, 7, 32, 25, 18, 18, 12, 28, 32, 22, 21, 26, 13, 25, 33, 20, 9, 27, 36, 40, 18, 24, 11, 17, 11, 26, 14, 20, 28, 21, 41, 25, 35, 18, 21, 30, 20, 17, 18, 27, 24, 25, 26, 17, 22, 14, 22, 27, 26, 24, 21, 30, 10, 14, 32, 30, 22, 17, 16, 23, 17, 29, 23, 16, 18, 22, 23, 25, 17, 24, 29, 15, 16, 46, 14, 27, 23, 24, 22, 20, 19, 7, 12, 21, 19, 19, 40, 13, 14, 26, 18, 27, 11, 31, 33, 21, 23, 9, 19, 20, 38, 26, 23, 18, 24, 26, 21, 17, 18, 24, 26, 24, 24, 37, 34, 15, 15, 17, 19, 25, 17, 9, 41, 24, 22, 18, 12, 37, 27, 12, 21, 29, 16, 24, 29, 23, 20, 33, 17, 17, 11, 19, 43, 31, 21, 42, 13, 31, 27, 35, 31, 13, 14, 38, 26, 24, 22, 32, 29, 62, 12, 9, 30, 15, 31, 28, 29, 13, 19, 13, 12, 12, 16, 27, 23, 32, 20, 10, 23, 27, 34, 15, 13, 19, 21, 11, 26, 28, 36, 14, 15, 18, 6, 30, 9, 18, 24, 32, 29, 25, 23, 22, 21, 20, 15, 26, 17, 29, 21, 20, 30, 18, 22, 21, 25, 38, 17, 11,

In [17]:
tags_test = list(data_test.POS)
pos_test = np.zeros((len(tags_test), ), dtype=int)
for i, val in enumerate(tags_test):
    pos_test[i] = tag2id[val]
len(pos_predict), len(pos_test), len(samples), len(word_test)

NameError: name 'pos_predict' is not defined

Somehow the output of HMM is in wrong size. Only use the shorter length to check the result.

In [13]:
def reportTest(y_pred, y_test):
    print("The accuracy is {}".format(accuracy_score(y_test, y_pred))) 
    print("The precision is {}".format(precision_score(y_test, y_pred, average='weighted'))) 
    print("The recall is {}".format(recall_score(y_test, y_pred, average='weighted'))) 
    print("The F1-Score is {}".format(f1_score(y_test, y_pred, average='weighted')))

min_length = min(len(pos_predict), len(pos_test))

reportTest(pos_predict[:min_length], pos_test[:min_length])

The accuracy is 0.9656062381551727
The precision is 0.9657832270028688
The recall is 0.9656062381551727
The F1-Score is 0.9655716883723663
